# Z-Algorithm

In [1]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

warnings.filterwarnings("ignore", message=".*tight_layout.*")


def safe_tight_layout(fig=None):
    """Call tight_layout suppressing the aspect-ratio incompatibility warning."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if fig is None:
            plt.tight_layout()
        else:
            fig.tight_layout()


plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "match": "#4CAF50",
    "mismatch": "#F44336",
    "current": "#FFC107",
    "skip": "#90CAF9",
    "default": "#E0E0E0",
    "pattern": "#BBDEFB",
    "border": "#7E57C2",
}


def draw_string_row(ax, y, string, label="", highlights=None, offset=0):
    """Draw a row of character boxes at vertical position y."""
    highlights = highlights or {}
    for i, ch in enumerate(string):
        color = highlights.get(i, COLORS["default"])
        rect = mpatches.FancyBboxPatch(
            (i + offset, y), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(i + offset + 0.45, y + 0.45, ch,
                ha="center", va="center", fontsize=14, fontweight="bold")
    if label:
        ax.text(offset - 0.3, y + 0.45, label,
                ha="right", va="center", fontsize=12, color="#555")


def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<", layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">", layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")

Helpers loaded (retina mode).


## Z-Algorithm

Visualize the Z-array: `Z[k]` = length of the longest substring starting at `k` that matches a prefix of `S`.

In [2]:
def compute_z(S):
    """Compute Z-array for string S."""
    n = len(S)
    Z = [0] * n
    Z[0] = n
    l, r = 0, 0
    for k in range(1, n):
        if k < r:
            Z[k] = min(r - k, Z[k - l])
        while k + Z[k] < n and S[Z[k]] == S[k + Z[k]]:
            Z[k] += 1
        if k + Z[k] > r:
            l, r = k, k + Z[k]
    return Z


def z_trace(S):
    """Step-by-step Z computation returning full state at each k."""
    n = len(S)
    Z = [0] * n
    Z[0] = n
    snapshots = []
    l, r = 0, 0
    for k in range(1, n):
        init_from_zbox = False
        init_val = 0
        if k < r:
            init_val = min(r - k, Z[k - l])
            Z[k] = init_val
            init_from_zbox = True

        explicit_comps = 0
        while k + Z[k] < n and S[Z[k]] == S[k + Z[k]]:
            Z[k] += 1
            explicit_comps += 1

        old_l, old_r = l, r
        if k + Z[k] > r:
            l, r = k, k + Z[k]

        snapshots.append({
            "k": k, "Z": list(Z), "l": l, "r": r,
            "old_l": old_l, "old_r": old_r,
            "init_from_zbox": init_from_zbox,
            "init_val": init_val,
            "explicit_comps": explicit_comps,
            "zbox_updated": (l != old_l or r != old_r),
        })
    return snapshots


def draw_z_step(S, step_idx):
    snapshots = z_trace(S)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    n = len(S)
    k = snap["k"]
    Z = snap["Z"]
    l, r = snap["l"], snap["r"]
    zk = Z[k]

    fig, axes = plt.subplots(2, 1, figsize=(max(n * 0.75, 10), 5.5),
                              gridspec_kw={"height_ratios": [3, 2]})

    # ── Panel 1: String + Z-box + highlights ──
    ax = axes[0]
    ax.set_xlim(-2.0, n + 1.0)
    ax.set_ylim(-1.5, 3.2)
    ax.set_aspect("equal")
    ax.axis("off")

    # index row
    for i in range(n):
        ax.text(i + 0.45, 2.5, str(i), ha="center", fontsize=7, color="#999")

    # string row with highlights
    s_hi = {}
    if zk > 0:
        for j in range(zk):
            s_hi[j] = COLORS["match"]
            s_hi[k + j] = COLORS["skip"]
        s_hi[k] = COLORS["current"]
    else:
        s_hi[k] = COLORS["current"]
    draw_string_row(ax, 1.2, S, label="S", highlights=s_hi)

    # k pointer below
    ax.annotate(f"k={k}", xy=(k + 0.45, 1.2), xytext=(k + 0.45, 0.6),
                fontsize=9, ha="center", color="#C62828", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#C62828", lw=1.5))

    # Z-box bracket above
    if r > 0:
        box_y = 2.8
        ax.plot([l, r - 0.1], [box_y, box_y], color=COLORS["border"], lw=2.5, solid_capstyle="round")
        ax.plot([l, l], [box_y - 0.1, box_y + 0.1], color=COLORS["border"], lw=1.5)
        ax.plot([r - 0.1, r - 0.1], [box_y - 0.1, box_y + 0.1], color=COLORS["border"], lw=1.5)
        ax.text((l + r) / 2, box_y + 0.2, f"Z-box [l={l}, r={r})",
                ha="center", fontsize=8, color=COLORS["border"], fontweight="bold")

    # brackets below for prefix and Z[k] match — on SEPARATE lines
    if zk > 0:
        # prefix bracket (higher line)
        y1 = -0.1
        ax.annotate("", xy=(0, y1), xytext=(zk - 0.1, y1),
                     arrowprops=dict(arrowstyle="<->", color=COLORS["match"], lw=1.5))
        ax.text(zk / 2, y1 - 0.35, f"prefix[0..{zk-1}]", ha="center",
                fontsize=8, color=COLORS["match"], fontweight="bold")

        # Z[k] bracket (lower line)
        y2 = -0.8
        ax.annotate("", xy=(k, y2), xytext=(k + zk - 0.1, y2),
                     arrowprops=dict(arrowstyle="<->", color="#1565C0", lw=2))
        ax.text(k + zk / 2, y2 - 0.35, f"Z[{k}]={zk}", ha="center",
                fontsize=9, color="#1565C0", fontweight="bold")

    # info text
    parts = []
    if snap["init_from_zbox"]:
        parts.append(f"Z-box reuse: min({snap['old_r']}-{k}, Z[{k-l}])={snap['init_val']}")
    if snap["explicit_comps"] > 0:
        parts.append(f"explicit comps: {snap['explicit_comps']}")
    if snap["zbox_updated"]:
        parts.append(f"Z-box: [{snap['old_l']},{snap['old_r']}) -> [{l},{r})")
    info = " | ".join(parts) if parts else f"Z[{k}]=0"

    ax.set_title(f"Z step {step_idx+1}/{len(snapshots)} | k={k} | Z[{k}]={zk}\n{info}",
                 fontsize=10, pad=6)

    # ── Panel 2: Full Z-array so far ──
    ax2 = axes[1]
    ax2.set_xlim(-2.0, n + 0.5)
    ax2.set_ylim(-0.3, 2.2)
    ax2.set_aspect("equal")
    ax2.axis("off")

    for i in range(n):
        computed = i == 0 or i <= k
        ax2.text(i + 0.45, 1.55, S[i], ha="center", fontsize=9, fontfamily="monospace",
                 color="#333" if computed else "#CCC")
        if i == k:
            color = COLORS["current"]
        elif computed and Z[i] > 0:
            color = "#E8F5E9"
        elif computed:
            color = "#FAFAFA"
        else:
            color = "#F0F0F0"
        rect = mpatches.FancyBboxPatch(
            (i, 0.3), 0.9, 0.9, boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555" if computed else "#CCC",
            linewidth=1.0 if computed else 0.5)
        ax2.add_patch(rect)
        val = str(Z[i]) if computed else "?"
        ax2.text(i + 0.45, 0.75, val, ha="center", va="center", fontsize=10,
                 fontweight="bold" if computed else "normal",
                 color="#333" if computed else "#BBB")
        ax2.text(i + 0.45, 0.05, str(i), ha="center", fontsize=6, color="#999")
    ax2.text(-0.3, 0.75, "Z", ha="right", va="center", fontsize=11, color="#555")
    ax2.text(-0.3, 1.55, "S", ha="right", va="center", fontsize=11, color="#555")

    safe_tight_layout()
    plt.show()


S_z_input = widgets.Text(value="aabxaabxcaabxaabxay", description="S:", layout=widgets.Layout(width="500px"))
k_slider = widgets.IntSlider(value=0, min=0, max=0, description="Step:", layout=widgets.Layout(display="none"))


def _update_z_max(*_):
    S = S_z_input.value
    if S and len(S) >= 2:
        k_slider.max = max(len(z_trace(S)) - 1, 0)

S_z_input.observe(_update_z_max, "value")
_update_z_max()


def _draw_z(S, step):
    if S and len(S) >= 2:
        draw_z_step(S, step)

out_z = widgets.interactive_output(_draw_z, {"S": S_z_input, "step": k_slider})
stepper_z = make_stepper(k_slider, "Step")
display(S_z_input, stepper_z, out_z)

Text(value='aabxaabxcaabxaabxay', description='S:', layout=Layout(width='500px'))

Output()